### Imports

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

In [2]:
def load_data(path: str | Path, **kwargs) -> pd.DataFrame:
    """
    Load CSV or Excel files.

    Parameters
    ----------
    path : str or Path
        Path to the dataset.

    Returns
    -------
    pd.DataFrame
    """
    path = Path(path)

    suffix = path.suffix.lower()

    if suffix == ".csv":
        try:
            return pd.read_csv(path, **kwargs)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding="latin1", **kwargs)

    elif suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path, **kwargs)

    else:
        raise ValueError(f"Unsupported file format: {suffix}")


def dataset_overview(df: pd.DataFrame) -> None:
    """Print a quick overview of a dataframe."""

    print("=" * 60)
    print(f"Rows:    {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print()

    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum().sort_values(ascending=False))

    print("\nMemory usage:")
    print(f"{df.memory_usage(deep=True).sum()/1024**2:.2f} MB")

In [3]:
df = load_data("data.csv")
dataset_overview(df)

Rows:    10
Columns: 22

company_id               str
ateco_sector             str
region                   str
legal_form               str
employees              int64
company_age            int64
total_assets           int64
current_assets         int64
total_liabilities      int64
equity                 int64
revenue                int64
ebitda                 int64
net_income             int64
cash                   int64
debt                   int64
equity_ratio         float64
current_ratio        float64
roa                  float64
roe                  float64
debt_to_equity       float64
asset_turnover       float64
target                   str
dtype: object

Missing values:
company_id           0
ateco_sector         0
region               0
legal_form           0
employees            0
company_age          0
total_assets         0
current_assets       0
total_liabilities    0
equity               0
revenue              0
ebitda               0
net_income           0
cash   

### Fitting and testing

In [4]:
def prepare_data(df, target):
    X = df.drop(columns=target)
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42, stratify=y)

    num_cols = X.select_dtypes(include="number").columns
    cat_cols = X.select_dtypes(exclude="number").columns

    return X_train, X_test, y_train, y_test, num_cols, cat_cols


def train_model(X_train, y_train, num_cols, cat_cols, classifier):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median"))
                ]),
                num_cols,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore"))
                ]),
                cat_cols,
            ),
        ]
    )

    model = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", classifier)
    ])

    model.fit(X_train, y_train)

    return model

In [5]:
target = "target" # name of the target column in dataset
X_train, X_test, y_train, y_test, num_cols, cat_cols = prepare_data(df, target)

In [6]:
model = train_model(X_train, y_train, num_cols, cat_cols, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
model = train_model(X_train, y_train, num_cols, cat_cols, RandomForestClassifier(random_state=42))
model = train_model(X_train, y_train, num_cols, cat_cols, HistGradientBoostingClassifier(random_state=42))

In [7]:
y_pred = model.predict(X_test)

In [8]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}\n")

print("Classification report")
print(classification_report(y_test, y_pred))

print("Confusion matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.600

Classification report
              precision    recall  f1-score   support

  Distressed       0.00      0.00      0.00         1
     Healthy       0.60      1.00      0.75         3
   Watchlist       0.00      0.00      0.00         1

    accuracy                           0.60         5
   macro avg       0.20      0.33      0.25         5
weighted avg       0.36      0.60      0.45         5

Confusion matrix
[[0 1 0]
 [0 3 0]
 [0 1 0]]


c:\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
